# Predicción y análisis de flujos de movilidad urbana en Valencia (2022-2024)

**Trabajo Fin de Máster — Máster Universitario en Big Data y Ciencia de Datos**

Universidad Internacional de Valencia (VIU)

Este cuaderno contiene el código empleado en el trabajo.

El flujo se organiza en cuatro bloques encadenados:

- Adquisición y construcción de la serie diaria de movilidad del municipio de Valencia (código INE 46250)
- Construcción de las variables externas de calendario y meteorología, e integración en una única serie
- Modelado predictivo comparado con SARIMA, SARIMAX, Prophet y LSTM, en dos esquemas de predicción y frente a modelos ingenuos de referencia
- Caracterización del episodio de la DANA mediante detección de anomalías y clasificación supervisada

Cada celda de código depende de las anteriores, por lo que el cuaderno debe ejecutarse de arriba abajo. Las celdas de las secciones 2 y 3 tardan varias horas y escriben ficheros de gran tamaño (16 GB).

## 1. Configuración

### 1.1. Librerías y rutas del proyecto

Los directorios de salida se crean si no existen.

In [ ]:
import json
import os
import random
import warnings
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent

RAW = BASE / "data" / "raw"
EXTERNAL = BASE / "data" / "external"
PROC = BASE / "data" / "processed"
CHECK = BASE / "data" / "checkpoints"
RES = PROC / "resultados_comparativa"
RES_B2 = PROC / "resultados_bloque2"
LSTM_DIR = PROC / "lstm"
FIG = BASE / "figuras"

for carpeta in [RAW, EXTERNAL, PROC, CHECK, RES, RES_B2, LSTM_DIR, FIG]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("Raiz del proyecto:", BASE)

### 1.2. Constantes del estudio

El municipio de Valencia se identifica por su código INE. La ventana de la DANA corresponde al intervalo definido por la AEMET entre el 28 de octubre y el 4 de noviembre de 2024.

In [ ]:
MUNICIPIO = "46250"
ANIOS = [2022, 2023, 2024]

FECHA_INICIO = "2022-01-01"
FECHA_FIN = "2024-12-31"

FIN_TRAIN = "2023-12-31"
INICIO_OBJ1 = "2024-01-01"
FIN_OBJ1 = "2024-10-27"
INICIO_OBJ2 = "2024-10-28"
FIN_OBJ2 = "2024-11-04"

OBJETIVOS = ["entradas_viajes", "salidas_viajes"]
EXTERNAS = ["temperatura", "precipitacion", "viento", "humedad_relativa",
            "es_festivo", "es_fallas", "es_dana"]
CALENDARIO = ["es_festivo", "es_fallas"]
ESTACIONALIDAD = 7

## 2. Adquisición de los datos de movilidad

Los datos proceden de los estudios de movilidad cotidiana del MITMA, elaborados a partir de la posición de las telefonías móviles. Se accede a ellos mediante la librería `pySpainMobility`, que gestiona la descarga desde el portal de datos abiertos.

### 2.1. Descarga de las matrices origen-destino

La descarga se realiza año a año, con el nivel de agregación de municipios. Cada llamada deja en disco los ficheros diarios comprimidos correspondientes al periodo solicitado.

In [ ]:
from pyspainmobility import Mobility

for anio in ANIOS:
    destino = RAW / f"movilidad_{anio}"
    destino.mkdir(parents=True, exist_ok=True)
    mobility = Mobility(
        version=2,
        zones="municipalities",
        start_date=f"{anio}-01-01",
        end_date=f"{anio}-12-31",
        backend="pandas",
        output_directory=str(destino),
    )
    mobility.get_od_data()
    print(f"Descarga de {anio} completada en {destino}")

### 2.2. Filtrado de los flujos del municipio de Valencia



In [ ]:
COLUMNAS_OD = ["fecha", "periodo", "origen", "destino", "distancia",
               "actividad_origen", "actividad_destino", "viajes", "viajes_km"]

for anio in ANIOS:
    carpeta = RAW / f"movilidad_{anio}"
    salida = PROC / f"movilidad_valencia_{anio}.csv"
    checkpoint = CHECK / f"checkpoint_filtrado_{anio}.txt"

    procesados = set(checkpoint.read_text().splitlines()) if checkpoint.exists() else set()
    archivos = sorted(carpeta.glob("*.csv.gz"))
    primero = not salida.exists()
    print(f"Anio {anio}: {len(archivos)} archivos, {len(procesados)} ya procesados")

    for i, archivo in enumerate(archivos, start=1):
        if archivo.name in procesados:
            continue
        try:
            df = pd.read_csv(archivo, sep="|", compression="gzip", dtype=str,
                             usecols=COLUMNAS_OD)
            mask = (df["origen"] == MUNICIPIO) | (df["destino"] == MUNICIPIO)
            df_val = df[mask]
            if len(df_val) > 0:
                df_val.to_csv(salida, mode="a", header=primero, index=False)
                primero = False
            with open(checkpoint, "a") as f:
                f.write(archivo.name + "\n")
            procesados.add(archivo.name)
            if i % 25 == 0:
                print(f"  {i}/{len(archivos)} archivos procesados")
        except Exception as e:
            print(f"  Error en {archivo.name}: {e}")

    print(f"Anio {anio} completado. Salida: {salida}")

### 2.3. Unión de los tres años en un único fichero de detalle

Los tres ficheros anuales se concatenan por bloques en `movilidad_valencia_2022_2024.csv`, que es el fichero de detalle origen-destino.

In [ ]:
salida_od = PROC / "movilidad_valencia_2022_2024.csv"
primero = True

for anio in ANIOS:
    ruta = PROC / f"movilidad_valencia_{anio}.csv"
    for trozo in pd.read_csv(ruta, dtype=str, chunksize=2_000_000):
        trozo.to_csv(salida_od, mode="w" if primero else "a",
                     header=primero, index=False)
        primero = False
    print(f"Anio {anio} anadido")

print("Fichero de detalle:", salida_od)

## 3. Construcción de la serie diaria de movilidad

### 3.1. Funciones de agregación y control de progreso

La agregación recorre el fichero de detalle por bloques de cinco millones de filas y produce cuatro salidas. La primera es la serie diaria de entradas y salidas de todo el periodo. Las otras tres conservan el detalle completo de los meses de septiembre a noviembre de 2024, 2023 y 2022, que sirven de referencia comparativa para el episodio.



In [ ]:
RUTA_CHECKPOINT_AGREGACION = CHECK / "checkpoint_agregacion.json"

DTYPES_OD = {
    "fecha": "int64",
    "periodo": "int8",
    "origen": "string",
    "destino": "string",
    "distancia": "string",
    "actividad_origen": "string",
    "actividad_destino": "string",
    "viajes": "float64",
    "viajes_km": "float64",
}

CHUNK_SIZE = 5_000_000
TOTAL_FILAS_ESTIMADO = 230_024_240

VENTANAS_DETALLE = {
    "movilidad_dana_2024.csv": (20240901, 20241130),
    "movilidad_referencia_2023.csv": (20230901, 20231130),
    "movilidad_referencia_2022.csv": (20220901, 20221130),
}


def cargar_checkpoint():
    if RUTA_CHECKPOINT_AGREGACION.exists():
        with open(RUTA_CHECKPOINT_AGREGACION) as f:
            datos = json.load(f)
        datos["agregado"] = {int(k): v for k, v in datos["agregado"].items()}
        return datos
    return {
        "chunks_procesados": 0,
        "agregado": {},
        "filas_escritas": {nombre: 0 for nombre in VENTANAS_DETALLE},
        "completado": False,
    }


def guardar_checkpoint(cp):
    datos = dict(cp)
    datos["agregado"] = {str(k): v for k, v in cp["agregado"].items()}
    with open(RUTA_CHECKPOINT_AGREGACION, "w") as f:
        json.dump(datos, f)


def actualizar_agregado(agregado, fecha, ev, ekm, sv, skm):
    if fecha not in agregado:
        agregado[fecha] = {"entradas_viajes": 0.0, "entradas_viajes_km": 0.0,
                           "salidas_viajes": 0.0, "salidas_viajes_km": 0.0}
    agregado[fecha]["entradas_viajes"] += ev
    agregado[fecha]["entradas_viajes_km"] += ekm
    agregado[fecha]["salidas_viajes"] += sv
    agregado[fecha]["salidas_viajes_km"] += skm


def escribir_bloque(df_bloque, ruta, primera_escritura):
    if len(df_bloque) == 0:
        return primera_escritura, 0
    df_bloque.to_csv(ruta, mode="w" if primera_escritura else "a",
                     header=primera_escritura, index=False)
    return False, len(df_bloque)

### 3.2. Agregación diaria de entradas y salidas

In [ ]:
cp = cargar_checkpoint()

if cp["completado"]:
    print("El proceso ya figura como completo en el punto de control.")
else:
    primeras = {nombre: (cp["filas_escritas"][nombre] == 0
                         and not (PROC / nombre).exists())
                for nombre in VENTANAS_DETALLE}

    lector = pd.read_csv(salida_od, dtype=DTYPES_OD, chunksize=CHUNK_SIZE)
    indice_chunk = 0

    for _ in range(cp["chunks_procesados"]):
        next(lector)
        indice_chunk += 1
    if indice_chunk:
        print(f"Saltados {indice_chunk} bloques ya procesados")

    for chunk in lector:
        entradas = chunk[chunk["destino"] == MUNICIPIO]
        salidas = chunk[chunk["origen"] == MUNICIPIO]
        ent_agr = entradas.groupby("fecha")[["viajes", "viajes_km"]].sum()
        sal_agr = salidas.groupby("fecha")[["viajes", "viajes_km"]].sum()

        for fecha in set(ent_agr.index) | set(sal_agr.index):
            ev = float(ent_agr.loc[fecha, "viajes"]) if fecha in ent_agr.index else 0.0
            ekm = float(ent_agr.loc[fecha, "viajes_km"]) if fecha in ent_agr.index else 0.0
            sv = float(sal_agr.loc[fecha, "viajes"]) if fecha in sal_agr.index else 0.0
            skm = float(sal_agr.loc[fecha, "viajes_km"]) if fecha in sal_agr.index else 0.0
            actualizar_agregado(cp["agregado"], int(fecha), ev, ekm, sv, skm)

        for nombre, (inicio, fin) in VENTANAS_DETALLE.items():
            bloque = chunk[(chunk["fecha"] >= inicio) & (chunk["fecha"] <= fin)]
            primeras[nombre], escritas = escribir_bloque(
                bloque, PROC / nombre, primeras[nombre])
            cp["filas_escritas"][nombre] += escritas

        indice_chunk += 1
        cp["chunks_procesados"] = indice_chunk
        guardar_checkpoint(cp)

        avance = min(100.0, indice_chunk * CHUNK_SIZE / TOTAL_FILAS_ESTIMADO * 100)
        print(f"Bloque {indice_chunk} procesado. Avance estimado {avance:.1f}%")

    df_diario = (pd.DataFrame([{"fecha": fecha, **valores}
                               for fecha, valores in cp["agregado"].items()])
                 .sort_values("fecha").reset_index(drop=True))
    df_diario.to_csv(PROC / "serie_diaria_valencia.csv", index=False)

    cp["completado"] = True
    guardar_checkpoint(cp)
    print(f"Serie diaria generada con {len(df_diario)} dias")

### 3.3. Verificación de las salidas generadas

In [ ]:
with open(RUTA_CHECKPOINT_AGREGACION) as f:
    cp = json.load(f)

print("Completado:", cp["completado"])
print("Bloques procesados:", cp["chunks_procesados"])
print("Fechas distintas en el agregado:", len(cp["agregado"]))
for nombre, filas in cp["filas_escritas"].items():
    print(f"{nombre}: {filas:,} filas")

for nombre in ["serie_diaria_valencia.csv", *VENTANAS_DETALLE]:
    ruta = PROC / nombre
    estado = f"{ruta.stat().st_size / 1024**2:.2f} MB" if ruta.exists() else "no existe"
    print(f"{nombre}: {estado}")

df_diaria = pd.read_csv(PROC / "serie_diaria_valencia.csv")
print("Filas:", len(df_diaria))
print("Rango:", df_diaria["fecha"].min(), "a", df_diaria["fecha"].max())

### 3.4. Identificación de días sin dato

El operador no publica datos para todas las fechas del periodo. Antes de reindexar conviene saber cuántos días faltan y cuáles son, porque condicionan tanto el tamaño de la ventana deslizante de la red LSTM.

In [ ]:
diaria = pd.read_csv(PROC / "serie_diaria_valencia.csv")
diaria["fecha"] = pd.to_datetime(diaria["fecha"], format="%Y%m%d")

calendario_completo = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="D")
faltantes = sorted(set(calendario_completo.date) - set(diaria["fecha"].dt.date))

print("Dias esperados:", len(calendario_completo))
print("Dias presentes:", len(diaria))
print("Dias faltantes:", len(faltantes))
for fecha in faltantes:
    print(fecha)

### 3.5. Reindexado al calendario completo

La serie se reindexa sobre los 1.096 días del periodo y se añade la variable indicador `es_faltante`, que marca las fechas sin dato. 

In [ ]:
reindexada = (diaria.set_index("fecha").sort_index()
              .reindex(pd.DatetimeIndex(calendario_completo, name="fecha")))
reindexada["es_faltante"] = reindexada["entradas_viajes"].isna().astype(int)
reindexada = reindexada.reset_index()

print("Dias:", len(reindexada))
print("Dias marcados como faltantes:", int(reindexada["es_faltante"].sum()))

reindexada.to_csv(PROC / "serie_diaria_valencia_reindexada.csv", index=False)
print("Guardado en:", PROC / "serie_diaria_valencia_reindexada.csv")

## 4. Variables externas

### 4.1. Calendario de indicadores

Se construyen tres variables indicador. La primera recoge los festivos nacionales, autonómicos de la Comunidad Valenciana y locales del municipio de Valencia. La segunda cubre el periodo de Fallas, del 1 al 19 de marzo. La tercera delimita la ventana del episodio de la DANA según el estudio de AEMET.

Los festivos se obtienen con `observed=False` para incluir también los que caen en domingo, evitando la asimetría de excluir domingos pero mantener sábados. Los dos festivos locales del municipio no están cubiertos por la librería, que solo llega al nivel autonómico, y se añaden de forma explícita.

In [ ]:
import holidays

fechas = pd.date_range(start=FECHA_INICIO, end=FECHA_FIN, freq="D")
calendario = pd.DataFrame({"fecha": fechas})

festivos_cv = holidays.Spain(subdiv="VC", years=ANIOS, observed=False)
calendario["es_festivo"] = calendario["fecha"].dt.date.isin(festivos_cv).astype(int)

festivos_locales_valencia = pd.to_datetime([
    "2022-01-22", "2022-04-25",
    "2023-04-17", "2023-04-24",
    "2024-01-22", "2024-04-08",
])
calendario.loc[calendario["fecha"].isin(festivos_locales_valencia), "es_festivo"] = 1

calendario["es_fallas"] = (
    (calendario["fecha"].dt.month == 3)
    & (calendario["fecha"].dt.day >= 1)
    & (calendario["fecha"].dt.day <= 19)
).astype(int)

calendario["es_dana"] = (
    (calendario["fecha"] >= pd.Timestamp(INICIO_OBJ2))
    & (calendario["fecha"] <= pd.Timestamp(FIN_OBJ2))
).astype(int)

print("Total dias:", len(calendario))
print("Dias festivos:", int(calendario["es_festivo"].sum()))
print("Dias de Fallas:", int(calendario["es_fallas"].sum()))
print("Dias de la ventana DANA:", int(calendario["es_dana"].sum()))

calendario.to_csv(PROC / "calendario_indicadores.csv", index=False)

### 4.2. Descarga de los datos meteorológicos

Los datos horarios proceden de Open-Meteo, con las coordenadas oficiales del municipio de Valencia. Se descargan cuatro variables continuas. 

In [ ]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession(".cache", expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

params = {
    "latitude": 39.4767,
    "longitude": -0.3744,
    "start_date": FECHA_INICIO,
    "end_date": FECHA_FIN,
    "hourly": ["temperature_2m", "precipitation",
               "wind_speed_10m", "relative_humidity_2m"],
    "timezone": "Europe/Madrid",
}

responses = openmeteo.weather_api(
    "https://archive-api.open-meteo.com/v1/archive", params=params)
response = responses[0]
hourly = response.Hourly()

df_meteo = pd.DataFrame({
    "fecha_hora": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left",
    ),
    "temperatura": hourly.Variables(0).ValuesAsNumpy(),
    "precipitacion": hourly.Variables(1).ValuesAsNumpy(),
    "viento": hourly.Variables(2).ValuesAsNumpy(),
    "humedad_relativa": hourly.Variables(3).ValuesAsNumpy(),
})
df_meteo["fecha_hora"] = df_meteo["fecha_hora"].dt.tz_convert("Europe/Madrid")

print("Filas:", len(df_meteo))
print("Rango:", df_meteo["fecha_hora"].min(), "a", df_meteo["fecha_hora"].max())

df_meteo.to_csv(EXTERNAL / "meteorologia_valencia.csv", index=False)

### 4.3. Agregación diaria de la meteorología

La serie horaria se agrega a resolución diaria. La temperatura, el viento y la humedad relativa se resumen por su media, mientras que la precipitación se acumula por suma.

In [ ]:
meteo_horaria = pd.read_csv(EXTERNAL / "meteorologia_valencia.csv")
meteo_horaria["fecha_hora"] = (pd.to_datetime(meteo_horaria["fecha_hora"], utc=True)
                               .dt.tz_convert("Europe/Madrid"))
meteo_horaria["fecha"] = meteo_horaria["fecha_hora"].dt.date

meteo_diaria = meteo_horaria.groupby("fecha").agg(
    temperatura=("temperatura", "mean"),
    precipitacion=("precipitacion", "sum"),
    viento=("viento", "mean"),
    humedad_relativa=("humedad_relativa", "mean"),
).reset_index()
meteo_diaria["fecha"] = pd.to_datetime(meteo_diaria["fecha"])

print("Dias agregados:", len(meteo_diaria))
print(meteo_diaria.describe().round(2))

meteo_diaria.to_csv(PROC / "meteo_diaria_agregada.csv", index=False)

## 5. Integración y partición de los datos

### 5.1. Unión de la serie de movilidad con las variables externas

La unión se hace por fecha y conserva las 1.096 filas de la serie de movilidad, que actúa como tabla principal.

In [ ]:
movilidad = pd.read_csv(PROC / "serie_diaria_valencia_reindexada.csv",
                        parse_dates=["fecha"])
calendario = pd.read_csv(PROC / "calendario_indicadores.csv", parse_dates=["fecha"])
meteo = pd.read_csv(PROC / "meteo_diaria_agregada.csv", parse_dates=["fecha"])

serie_completa = (movilidad
                  .merge(calendario, on="fecha", how="left")
                  .merge(meteo, on="fecha", how="left")
                  .sort_values("fecha")
                  .reset_index(drop=True))

print("Filas:", len(serie_completa))
print("Columnas:", list(serie_completa.columns))
print(serie_completa.isnull().sum())

serie_completa.to_csv(PROC / "serie_diaria_con_externas.csv", index=False)

### 5.2. Partición cronológica en entrenamiento y evaluación

El tramo de entrenamiento abarca los dos primeros años completos. La evaluación se divide en dos objetivos, el primero cubre la mayor parte de 2024 en condiciones normales y el segundo se limita a los ocho días de la ventana del episodio.

In [ ]:
serie = pd.read_csv(PROC / "serie_diaria_con_externas.csv", parse_dates=["fecha"])
serie = serie.sort_values("fecha").reset_index(drop=True)

train = serie[serie["fecha"] <= FIN_TRAIN].reset_index(drop=True)
test_obj1 = serie[(serie["fecha"] >= INICIO_OBJ1)
                  & (serie["fecha"] <= FIN_OBJ1)].reset_index(drop=True)
test_obj2 = serie[(serie["fecha"] >= INICIO_OBJ2)
                  & (serie["fecha"] <= FIN_OBJ2)].reset_index(drop=True)

print(f"Train: {len(train)} dias ({train['fecha'].min().date()} a {train['fecha'].max().date()})")
print(f"Objetivo 1: {len(test_obj1)} dias ({test_obj1['fecha'].min().date()} a {test_obj1['fecha'].max().date()})")
print(f"Objetivo 2: {len(test_obj2)} dias ({test_obj2['fecha'].min().date()} a {test_obj2['fecha'].max().date()})")

todas = pd.concat([train["fecha"], test_obj1["fecha"], test_obj2["fecha"]])
print("Fechas duplicadas entre tramos:", int(todas.duplicated().sum()))
print("Dias posteriores al 4 de noviembre:", int((serie["fecha"] > FIN_OBJ2).sum()))

train.to_csv(PROC / "train_2022_2023.csv", index=False)
test_obj1.to_csv(PROC / "test_objetivo1_2024.csv", index=False)
test_obj2.to_csv(PROC / "test_objetivo2_dana.csv", index=False)

## 6. Análisis exploratorio

### 6.1. Estadísticos descriptivos y correlaciones

Se calculan sobre las observaciones válidas, esto es, excluyendo los días marcados como faltantes. Las correlaciones con las variables meteorológicas se calculan sobre el flujo total, suma de entradas y salidas.

In [ ]:
df = pd.read_csv(PROC / "serie_diaria_con_externas.csv", parse_dates=["fecha"])
validos = df[df["es_faltante"] == 0].copy()
validos["flujo_total"] = validos["entradas_viajes"] + validos["salidas_viajes"]

print(f"Observaciones validas: {len(validos)} de {len(df)}")
print(f"Media entradas: {validos['entradas_viajes'].mean():,.0f}"
      f"   desviacion: {validos['entradas_viajes'].std():,.0f}")
print(f"Media salidas:  {validos['salidas_viajes'].mean():,.0f}"
      f"   desviacion: {validos['salidas_viajes'].std():,.0f}")
print(f"Correlacion entradas-salidas: {validos['entradas_viajes'].corr(validos['salidas_viajes']):.4f}")

for variable in ["precipitacion", "temperatura", "viento", "humedad_relativa"]:
    print(f"Correlacion flujo total - {variable}: "
          f"{validos['flujo_total'].corr(validos[variable]):.3f}")

### 6.2. Serie diaria completa con la ventana del episodio



In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["fecha"], df["entradas_viajes"], linewidth=0.8, label="Entradas")
ax.plot(df["fecha"], df["salidas_viajes"], linewidth=0.8, label="Salidas")
ax.axvspan(pd.Timestamp(INICIO_OBJ2), pd.Timestamp(FIN_OBJ2),
           color="red", alpha=0.15, label="Episodio DANA")

faltantes_fechas = df.loc[df["es_faltante"] == 1, "fecha"]
for i, f in enumerate(faltantes_fechas):
    ax.axvline(f, color="black", alpha=0.7, linewidth=1.2,
               label="Dias faltantes" if i == 0 else None)

ax.set_xlabel("Fecha")
ax.set_ylabel("Viajes diarios")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "fig_serie_diaria_2022_2024.png", dpi=200)
plt.show()

### 6.3. Patrón semanal medio

In [ ]:
dias = ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes", "Sabado", "Domingo"]
patron = validos.groupby(validos["fecha"].dt.dayofweek)[OBJETIVOS].mean()

fig, ax = plt.subplots(figsize=(9, 5))
x = range(7)
ax.bar([i - 0.2 for i in x], patron["entradas_viajes"], width=0.4, label="Entradas")
ax.bar([i + 0.2 for i in x], patron["salidas_viajes"], width=0.4, label="Salidas")
ax.set_xticks(list(x))
ax.set_xticklabels(dias, rotation=30)
ax.set_ylabel("Viajes diarios medios")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "fig_patron_semanal.png", dpi=200)
plt.show()

### 6.4. Matriz de correlación entre flujos y variables externas

In [ ]:
COLUMNAS_CORR = OBJETIVOS + EXTERNAS
ETIQUETAS = ["Entradas", "Salidas", "Temperatura", "Precipitación", "Viento",
             "Humedad relativa", "Es festivo", "Es Fallas", "Es DANA"]

corr = validos[COLUMNAS_CORR].corr()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(ETIQUETAS)))
ax.set_yticks(range(len(ETIQUETAS)))
ax.set_xticklabels(ETIQUETAS, rotation=45, ha="right")
ax.set_yticklabels(ETIQUETAS)
for i in range(len(ETIQUETAS)):
    for j in range(len(ETIQUETAS)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(FIG / "fig_matriz_correlacion.png", dpi=200)
plt.show()

print(corr.round(3))

### 6.5. Comprobación de la similitud entre entradas y salidas



In [ ]:
val = df.dropna(subset=OBJETIVOS)
dif = val["entradas_viajes"] - val["salidas_viajes"]
rel = (dif / val["entradas_viajes"]) * 100

print("Observaciones validas:", len(val))
print(f"Diferencia diaria media absoluta: {dif.abs().mean():,.0f} viajes")
print(f"Diferencia diaria relativa media: {rel.abs().mean():.4f}%")
print(f"Diferencia diaria relativa maxima: {rel.abs().max():.4f}%")
print("Dias con diferencia relativa inferior al 0,5%:", int((rel.abs() < 0.5).sum()))
print(f"Correlacion: {val['entradas_viajes'].corr(val['salidas_viajes']):.6f}")

## 7. Descomposición en flujos internos y externos

### 7.1. Separación de los tres agregados diarios

A partir del detalle origen-destino se separan los viajes internos al municipio, las entradas procedentes de otros municipios y las salidas hacia otros municipios. Esta separación permite comprobar si los viajes internos están contabilizados en las dos series.

In [ ]:
acumulado = []
for trozo in pd.read_csv(salida_od,
                         usecols=["fecha", "origen", "destino", "viajes"],
                         dtype={"fecha": "int64", "origen": "string",
                                "destino": "string", "viajes": "float64"},
                         chunksize=5_000_000):
    es_origen = trozo["origen"] == MUNICIPIO
    es_destino = trozo["destino"] == MUNICIPIO
    trozo["internos"] = trozo["viajes"].where(es_origen & es_destino, 0.0)
    trozo["entradas_externas"] = trozo["viajes"].where(~es_origen & es_destino, 0.0)
    trozo["salidas_externas"] = trozo["viajes"].where(es_origen & ~es_destino, 0.0)
    acumulado.append(trozo.groupby("fecha")[
        ["internos", "entradas_externas", "salidas_externas"]].sum())

agregados = pd.concat(acumulado).groupby(level=0).sum()
agregados.index = pd.to_datetime(agregados.index, format="%Y%m%d")
agregados.index.name = "fecha"

print(agregados.mean().round(0).to_string())
print(f"Peso de los internos sobre el total: "
      f"{agregados['internos'].mean() / agregados.sum(axis=1).mean() * 100:.1f}%")
print(f"Correlacion entradas externas / salidas externas: "
      f"{agregados['entradas_externas'].corr(agregados['salidas_externas']):.4f}")

serie_ref = pd.read_csv(PROC / "serie_diaria_con_externas.csv",
                        parse_dates=["fecha"]).set_index("fecha")
comp = agregados.join(serie_ref[OBJETIVOS], how="inner")
comp["internos_mas_entradas"] = comp["internos"] + comp["entradas_externas"]
print(f"Diferencia media frente a internos mas entradas externas: "
      f"{(comp['entradas_viajes'] - comp['internos_mas_entradas']).abs().mean():,.1f}")
print(f"Diferencia media frente a solo entradas externas: "
      f"{(comp['entradas_viajes'] - comp['entradas_externas']).abs().mean():,.0f}")

agregados.to_csv(PROC / "agregados_od_separados.csv")

### 7.2. Construcción de la serie de flujos externos

Se reconstruye una serie paralela que sustituye los flujos totales por los flujos de conexión con otros municipios, conservando las variables externas y la marca de días faltantes de la serie original.

In [ ]:
serie_base = pd.read_csv(PROC / "serie_diaria_con_externas.csv",
                         parse_dates=["fecha"]).set_index("fecha").asfreq("D")

externa = serie_base.drop(columns=OBJETIVOS).join(
    agregados[["entradas_externas", "salidas_externas"]].rename(
        columns={"entradas_externas": "entradas_viajes",
                 "salidas_externas": "salidas_viajes"}))
externa.loc[externa["es_faltante"] == 1, OBJETIVOS] = np.nan
externa = externa.asfreq("D")
externa.to_csv(PROC / "serie_diaria_externa.csv")

print("Filas:", len(externa), " faltantes:", int(externa["es_faltante"].sum()))
print(f"Media de entradas externas: {externa['entradas_viajes'].mean():,.0f}")
print(f"Correlacion entradas/salidas externas: "
      f"{externa['entradas_viajes'].corr(externa['salidas_viajes']):.4f}")

## 8. Modelado predictivo. Esquema A

En este primer esquema el modelo se ajusta una sola vez sobre el tramo de entrenamiento y proyecta el año 2024 completo sin recibir ninguna observación posterior.

### 8.1. SARIMA

El orden de diferenciación regular se fija mediante el contraste aumentado de Dickey-Fuller sobre la serie de entrenamiento, y la diferenciación estacional se fija en uno por la estacionalidad semanal. La búsqueda en malla se limita entonces a los órdenes autorregresivos y de media móvil, porque el criterio de información de Akaike no es comparable entre modelos con órdenes de diferenciación distintos.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX

RUTA_CHECKPOINT_ARIMA = CHECK / "checkpoint_grid_arima.json"
D_ESTACIONAL = 1

train = pd.read_csv(PROC / "train_2022_2023.csv",
                    parse_dates=["fecha"]).set_index("fecha").asfreq("D")
test1 = pd.read_csv(PROC / "test_objetivo1_2024.csv",
                    parse_dates=["fecha"]).set_index("fecha").asfreq("D")
test2 = pd.read_csv(PROC / "test_objetivo2_dana.csv",
                    parse_dates=["fecha"]).set_index("fecha").asfreq("D")
test_total = pd.concat([test1, test2])
n_pasos = len(test_total)


def metricas(y_real, y_pred):
    mask = y_real.notna() & y_pred.notna()
    y, yhat = y_real[mask], y_pred[mask]
    return (float(np.sqrt(np.mean((y - yhat) ** 2))),
            float(np.mean(np.abs(y - yhat))),
            float(np.mean(np.abs((y - yhat) / y)) * 100))


ordenes_d = {}
for var in OBJETIVOS:
    pvalor = adfuller(train[var].dropna(), autolag="AIC")[1]
    ordenes_d[var] = 0 if pvalor < 0.05 else 1
    print(f"{var}: p-valor ADF={pvalor:.4f} -> d={ordenes_d[var]}")

malla = list(product(range(3), range(3), range(2), range(2)))

if RUTA_CHECKPOINT_ARIMA.exists():
    with open(RUTA_CHECKPOINT_ARIMA) as f:
        resultados_grid = json.load(f)
else:
    resultados_grid = {}

mejores_ordenes_arima = {}
for var in OBJETIVOS:
    d = ordenes_d[var]
    evaluadas = 0
    for combo in malla:
        p, q, P, Q = combo
        k = f"{var}|{combo}"
        if k in resultados_grid:
            continue
        try:
            res = SARIMAX(train[var], order=(p, d, q),
                          seasonal_order=(P, D_ESTACIONAL, Q, ESTACIONALIDAD),
                          enforce_stationarity=False,
                          enforce_invertibility=False).fit(disp=False)
            resultados_grid[k] = res.aic
        except Exception:
            resultados_grid[k] = None
        evaluadas += 1
        if evaluadas % 10 == 0:
            with open(RUTA_CHECKPOINT_ARIMA, "w") as f:
                json.dump(resultados_grid, f)
            print(f"{var}: {evaluadas} combinaciones evaluadas")
    with open(RUTA_CHECKPOINT_ARIMA, "w") as f:
        json.dump(resultados_grid, f)
    validas = {k: v for k, v in resultados_grid.items()
               if k.startswith(var + "|") and v is not None}
    mejor = min(validas, key=validas.get)
    p, q, P, Q = eval(mejor.split("|")[1])
    mejores_ordenes_arima[var] = (p, d, q, P, D_ESTACIONAL, Q)
    print(f"Mejor orden {var}: ({p},{d},{q})({P},{D_ESTACIONAL},{Q},7)  AIC={validas[mejor]:.1f}")

filas_metricas = []
predicciones = pd.DataFrame(index=test_total.index)
for var in OBJETIVOS:
    p, d, q, P, D, Q = mejores_ordenes_arima[var]
    res = SARIMAX(train[var], order=(p, d, q),
                  seasonal_order=(P, D, Q, ESTACIONALIDAD),
                  enforce_stationarity=False,
                  enforce_invertibility=False).fit(disp=False)
    pred = res.forecast(steps=n_pasos)
    pred.index = test_total.index
    predicciones[f"pred_{var}"] = pred
    for nombre_obj, test_df in [("objetivo1", test1), ("objetivo2", test2)]:
        rmse, mae, mape_v = metricas(test_df[var], pred.loc[test_df.index])
        filas_metricas.append({"modelo": "SARIMA", "objetivo": nombre_obj,
                               "variable": var, "variante": "unica",
                               "RMSE": rmse, "MAE": mae, "MAPE": mape_v})
        print(f"{var} {nombre_obj}: RMSE={rmse:,.0f} MAE={mae:,.0f} MAPE={mape_v:.2f}%")

predicciones.to_csv(RES / "predicciones_arima.csv")
pd.DataFrame(filas_metricas).to_csv(RES / "metricas_arima.csv", index=False)
with open(RES / "ordenes_arima.json", "w") as f:
    json.dump({k: list(v) for k, v in mejores_ordenes_arima.items()}, f)

### 8.2. SARIMAX

Se replica el procedimiento anterior incorporando las variables externas de calendario y meteorología. El Objetivo 2 se evalúa en dos variantes, una sin conocimiento previo del episodio y otra con conocimiento previo.

In [ ]:
RUTA_CHECKPOINT_SARIMAX = CHECK / "checkpoint_grid_sarimax.json"

exog_train = train[EXTERNAS]
exog_test_con = test_total[EXTERNAS].copy()
exog_test_sin = exog_test_con.copy()
exog_test_sin["es_dana"] = 0

if RUTA_CHECKPOINT_SARIMAX.exists():
    with open(RUTA_CHECKPOINT_SARIMAX) as f:
        resultados_grid = json.load(f)
else:
    resultados_grid = {}

mejores_ordenes_sarimax = {}
for var in OBJETIVOS:
    d = ordenes_d[var]
    evaluadas = 0
    for combo in malla:
        p, q, P, Q = combo
        k = f"{var}|{combo}"
        if k in resultados_grid:
            continue
        try:
            res = SARIMAX(train[var], exog=exog_train, order=(p, d, q),
                          seasonal_order=(P, D_ESTACIONAL, Q, ESTACIONALIDAD),
                          enforce_stationarity=False,
                          enforce_invertibility=False).fit(disp=False)
            resultados_grid[k] = res.aic
        except Exception:
            resultados_grid[k] = None
        evaluadas += 1
        if evaluadas % 10 == 0:
            with open(RUTA_CHECKPOINT_SARIMAX, "w") as f:
                json.dump(resultados_grid, f)
            print(f"{var}: {evaluadas} combinaciones evaluadas")
    with open(RUTA_CHECKPOINT_SARIMAX, "w") as f:
        json.dump(resultados_grid, f)
    validas = {k: v for k, v in resultados_grid.items()
               if k.startswith(var + "|") and v is not None}
    mejor = min(validas, key=validas.get)
    p, q, P, Q = eval(mejor.split("|")[1])
    mejores_ordenes_sarimax[var] = (p, d, q, P, D_ESTACIONAL, Q)
    print(f"Mejor orden {var}: ({p},{d},{q})({P},{D_ESTACIONAL},{Q},7)  AIC={validas[mejor]:.1f}")

filas_metricas = []
predicciones = pd.DataFrame(index=test_total.index)
for var in OBJETIVOS:
    p, d, q, P, D, Q = mejores_ordenes_sarimax[var]
    res = SARIMAX(train[var], exog=exog_train, order=(p, d, q),
                  seasonal_order=(P, D, Q, ESTACIONALIDAD),
                  enforce_stationarity=False,
                  enforce_invertibility=False).fit(disp=False)
    print(f"{var} coeficiente es_dana: {res.params.get('es_dana', np.nan):.4f}"
          f"   p-valor: {res.pvalues.get('es_dana', np.nan):.4f}")
    for variante, exog_t in [("sin_conocimiento", exog_test_sin),
                             ("con_conocimiento", exog_test_con)]:
        pred = res.forecast(steps=n_pasos, exog=exog_t)
        pred.index = test_total.index
        predicciones[f"pred_{var}_{variante}"] = pred
        for nombre_obj, test_df in [("objetivo1", test1), ("objetivo2", test2)]:
            rmse, mae, mape_v = metricas(test_df[var], pred.loc[test_df.index])
            filas_metricas.append({"modelo": "SARIMAX", "objetivo": nombre_obj,
                                   "variable": var, "variante": variante,
                                   "RMSE": rmse, "MAE": mae, "MAPE": mape_v})
            print(f"{var} {nombre_obj} {variante}: "
                  f"RMSE={rmse:,.0f} MAE={mae:,.0f} MAPE={mape_v:.2f}%")

predicciones.to_csv(RES / "predicciones_sarimax.csv")
pd.DataFrame(filas_metricas).to_csv(RES / "metricas_sarimax.csv", index=False)
with open(RES / "ordenes_sarimax.json", "w") as f:
    json.dump({k: list(v) for k, v in mejores_ordenes_sarimax.items()}, f)

### 8.3. Prophet

Prophet se ajusta con estacionalidades anual y semanal explícitas y con las mismas variables externas incorporadas como regresores adicionales, en paralelo al diseño del SARIMAX.

In [ ]:
from prophet import Prophet

train_p = pd.read_csv(PROC / "train_2022_2023.csv", parse_dates=["fecha"])
test1_p = pd.read_csv(PROC / "test_objetivo1_2024.csv", parse_dates=["fecha"])
test2_p = pd.read_csv(PROC / "test_objetivo2_dana.csv", parse_dates=["fecha"])
test_total_p = pd.concat([test1_p, test2_p]).reset_index(drop=True)

filas_metricas = []
predicciones = pd.DataFrame({"fecha": test_total_p["fecha"]})

for var in OBJETIVOS:
    df_train = (train_p[["fecha", var] + EXTERNAS]
                .rename(columns={"fecha": "ds", var: "y"}).dropna(subset=["y"]))
    modelo = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                     daily_seasonality=False)
    for reg in EXTERNAS:
        modelo.add_regressor(reg)
    modelo.fit(df_train)

    futuro_con = test_total_p[["fecha"] + EXTERNAS].rename(columns={"fecha": "ds"})
    futuro_sin = futuro_con.copy()
    futuro_sin["es_dana"] = 0

    for variante, futuro in [("sin_conocimiento", futuro_sin),
                             ("con_conocimiento", futuro_con)]:
        pred = modelo.predict(futuro)["yhat"].reset_index(drop=True)
        predicciones[f"pred_{var}_{variante}"] = pred
        for nombre_obj, test_df in [("objetivo1", test1_p), ("objetivo2", test2_p)]:
            idx = test_total_p["fecha"].isin(test_df["fecha"])
            rmse, mae, mape_v = metricas(
                test_total_p.loc[idx, var].reset_index(drop=True),
                pred[idx.values].reset_index(drop=True))
            filas_metricas.append({"modelo": "Prophet", "objetivo": nombre_obj,
                                   "variable": var, "variante": variante,
                                   "RMSE": rmse, "MAE": mae, "MAPE": mape_v})
            print(f"{var} {nombre_obj} {variante}: "
                  f"RMSE={rmse:,.0f} MAE={mae:,.0f} MAPE={mape_v:.2f}%")

predicciones.to_csv(RES / "predicciones_prophet.csv", index=False)
pd.DataFrame(filas_metricas).to_csv(RES / "metricas_prophet.csv", index=False)

## 9. Red neuronal recurrente LSTM

### 9.1. Selección del tamaño de la ventana deslizante

La red trabaja con ventanas de días consecutivos, de modo que cualquier día sin dato invalida todas las secuencias que lo contienen. Se evalúa el porcentaje de secuencias perdidas para cuatro tamaños candidatos, con el fin de escoger un compromiso entre memoria temporal y aprovechamiento de los datos disponibles.

In [ ]:
train_v = pd.read_csv(PROC / "train_2022_2023.csv", parse_dates=["fecha"])
faltante = train_v["es_faltante"].values

resultados = []
for ventana in [7, 14, 21, 30]:
    total_posibles = len(train_v) - ventana
    perdidas = sum(1 for i in range(total_posibles)
                   if faltante[i:i + ventana].sum() > 0 or faltante[i + ventana] > 0)
    resultados.append({
        "ventana_dias": ventana,
        "secuencias_posibles": total_posibles,
        "secuencias_validas": total_posibles - perdidas,
        "secuencias_perdidas": perdidas,
        "pct_perdida": round(perdidas / total_posibles * 100, 2),
    })

print(pd.DataFrame(resultados).to_string(index=False))

### 9.2. Entrenamiento y evaluación

La red recibe nueve canales de entrada, las dos series objetivo, las cuatro variables meteorológicas y las tres variables indicador de calendario. El escalado se ajusta únicamente con el tramo de entrenamiento para evitar la fuga de información (*data leakage*), la partición interna de validación es cronológica y la predicción es a un paso con un contexto observado de catorce días. La semilla se fija para garantizar la reproducibilidad.

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

VENTANA = 14
CANALES = OBJETIVOS + ["temperatura", "precipitacion", "viento", "humedad_relativa",
                       "es_festivo", "es_fallas", "es_dana"]

df_lstm = pd.read_csv(PROC / "serie_diaria_con_externas.csv",
                      parse_dates=["fecha"]).set_index("fecha").asfreq("D")
train_lstm = df_lstm.loc[:FIN_TRAIN]

scaler_x = MinMaxScaler().fit(train_lstm[CANALES].dropna())
scaler_y = MinMaxScaler().fit(train_lstm[OBJETIVOS].dropna())

X_full = pd.DataFrame(scaler_x.transform(df_lstm[CANALES]),
                      index=df_lstm.index, columns=CANALES)
y_full = pd.DataFrame(scaler_y.transform(df_lstm[OBJETIVOS]),
                      index=df_lstm.index, columns=OBJETIVOS)


def construir_ventanas(fechas_objetivo):
    X, y, fechas = [], [], []
    for fecha in fechas_objetivo:
        inicio = fecha - pd.Timedelta(days=VENTANA)
        contexto = X_full.loc[inicio:fecha - pd.Timedelta(days=1)]
        objetivo = y_full.loc[fecha]
        if len(contexto) < VENTANA or contexto.isna().any().any() or objetivo.isna().any():
            continue
        X.append(contexto.values)
        y.append(objetivo.values)
        fechas.append(fecha)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), fechas


fechas_train = train_lstm.index[
    train_lstm.index >= train_lstm.index[0] + pd.Timedelta(days=VENTANA)]
X_train_all, y_train_all, _ = construir_ventanas(fechas_train)
n_val = int(len(X_train_all) * 0.15)
X_train, y_train = X_train_all[:-n_val], y_train_all[:-n_val]
X_val, y_val = X_train_all[-n_val:], y_train_all[-n_val:]
print(f"Ventanas validas: {len(X_train_all)}  train: {len(X_train)}  validacion: {len(X_val)}")

fechas_test = df_lstm.loc[INICIO_OBJ1:FIN_OBJ2].index
X_test, y_test, fechas_test_validas = construir_ventanas(fechas_test)


class ModeloLSTM(nn.Module):
    def __init__(self, n_canales):
        super().__init__()
        self.lstm = nn.LSTM(n_canales, 32, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.densa = nn.Linear(32, 16)
        self.relu = nn.ReLU()
        self.salida = nn.Linear(16, 2)

    def forward(self, x):
        out, _ = self.lstm(x)
        h = self.dropout(out[:, -1, :])
        return self.salida(self.relu(self.densa(h)))


modelo = ModeloLSTM(len(CANALES))
optimizador = torch.optim.Adam(modelo.parameters())
criterio = nn.MSELoss()

Xt, yt = torch.tensor(X_train), torch.tensor(y_train)
Xv, yv = torch.tensor(X_val), torch.tensor(y_val)

PACIENCIA = 15
mejor_perdida = float("inf")
mejor_epoca = 0
epocas_sin_mejora = 0
mejor_estado = None
generador = torch.Generator().manual_seed(SEED)

for epoca in range(1, 501):
    modelo.train()
    indices = torch.randperm(len(Xt), generator=generador)
    for i in range(0, len(indices), 32):
        lote = indices[i:i + 32]
        optimizador.zero_grad()
        perdida = criterio(modelo(Xt[lote]), yt[lote])
        perdida.backward()
        optimizador.step()
    modelo.eval()
    with torch.no_grad():
        perdida_val = criterio(modelo(Xv), yv).item()
    if perdida_val < mejor_perdida:
        mejor_perdida, mejor_epoca = perdida_val, epoca
        mejor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
        epocas_sin_mejora = 0
    else:
        epocas_sin_mejora += 1
    if epocas_sin_mejora >= PACIENCIA:
        break

modelo.load_state_dict(mejor_estado)
print(f"Parada en la epoca {epoca}, mejor epoca {mejor_epoca}, "
      f"perdida de validacion {mejor_perdida:.4f}")

modelo.eval()
with torch.no_grad():
    pred_escalada = modelo(torch.tensor(X_test)).numpy()
pred = scaler_y.inverse_transform(pred_escalada)
real = scaler_y.inverse_transform(y_test)

predicciones = pd.DataFrame(pred, columns=[f"pred_{v}" for v in OBJETIVOS],
                            index=pd.DatetimeIndex(fechas_test_validas, name="fecha"))
predicciones[[f"real_{v}" for v in OBJETIVOS]] = real

filas_metricas = []
for var in OBJETIVOS:
    for nombre_obj, mascara in [
            ("objetivo1", predicciones.index < INICIO_OBJ2),
            ("objetivo2", predicciones.index >= INICIO_OBJ2)]:
        y_r = predicciones.loc[mascara, f"real_{var}"]
        y_p = predicciones.loc[mascara, f"pred_{var}"]
        rmse, mae, mape_v = metricas(y_r, y_p)
        filas_metricas.append({"modelo": "LSTM", "objetivo": nombre_obj,
                               "variable": var, "variante": "unica",
                               "RMSE": rmse, "MAE": mae, "MAPE": mape_v})
        print(f"{var} {nombre_obj}: RMSE={rmse:,.0f} MAE={mae:,.0f} MAPE={mape_v:.2f}%")

predicciones.to_csv(RES / "predicciones_lstm.csv")
pd.DataFrame(filas_metricas).to_csv(RES / "metricas_lstm.csv", index=False)
torch.save(mejor_estado, LSTM_DIR / "modelo_lstm.pt")

## 10. Modelado predictivo. Esquema B, predicción a un día vista

La red LSTM predice cada día a partir de las catorce jornadas anteriores observadas, mientras que los modelos de la familia SARIMA proyectaban el año completo sin realimentación. Comparar ambos en esas condiciones sería asimétrico y favorecería artificialmente a la red.

Por tanto se reproduce el esquema de un día vista también para SARIMA y SARIMAX. El modelo se ajusta una única vez sobre el tramo de entrenamiento y después se le van incorporando las observaciones reales día a día, sin volver a estimar los parámetros. Se reutilizan los órdenes ya seleccionados por el criterio de Akaike.

### 10.1. SARIMA y SARIMAX a un día vista

In [ ]:
train = pd.read_csv(PROC / "train_2022_2023.csv",
                    parse_dates=["fecha"]).set_index("fecha").asfreq("D")
test1 = pd.read_csv(PROC / "test_objetivo1_2024.csv",
                    parse_dates=["fecha"]).set_index("fecha").asfreq("D")
test2 = pd.read_csv(PROC / "test_objetivo2_dana.csv",
                    parse_dates=["fecha"]).set_index("fecha").asfreq("D")
test_total = pd.concat([test1, test2])

with open(RES / "ordenes_arima.json") as f:
    ordenes_arima = json.load(f)
with open(RES / "ordenes_sarimax.json") as f:
    ordenes_sarimax = json.load(f)


def prediccion_un_dia(var, orden, usar_exog):
    p, d, q, P, D, Q = orden
    kwargs = dict(order=(p, d, q), seasonal_order=(P, D, Q, ESTACIONALIDAD),
                  enforce_stationarity=False, enforce_invertibility=False)
    if usar_exog:
        estado = SARIMAX(train[var], exog=train[EXTERNAS], **kwargs).fit(disp=False)
    else:
        estado = SARIMAX(train[var], **kwargs).fit(disp=False)

    predicciones = []
    for fecha in test_total.index:
        if usar_exog:
            exog_dia = test_total.loc[[fecha], EXTERNAS]
            paso = estado.forecast(steps=1, exog=exog_dia)
        else:
            paso = estado.forecast(steps=1)
        predicciones.append(float(paso.iloc[0]))
        nuevo_y = test_total.loc[[fecha], var]
        if usar_exog:
            estado = estado.append(nuevo_y, exog=exog_dia, refit=False)
        else:
            estado = estado.append(nuevo_y, refit=False)
    return pd.Series(predicciones, index=test_total.index)


filas = []
predicciones = pd.DataFrame(index=test_total.index)

for nombre, ordenes, usar_exog in [("SARIMA", ordenes_arima, False),
                                   ("SARIMAX", ordenes_sarimax, True)]:
    for var in OBJETIVOS:
        pred = prediccion_un_dia(var, ordenes[var], usar_exog)
        predicciones[f"pred_{nombre}_{var}"] = pred
        for nombre_obj, test_df in [("objetivo1", test1), ("objetivo2", test2)]:
            rmse, mae, mape_v = metricas(test_df[var], pred.loc[test_df.index])
            filas.append({"modelo": nombre, "esquema": "un_dia",
                          "objetivo": nombre_obj, "variable": var,
                          "RMSE": rmse, "MAE": mae, "MAPE": mape_v})
            print(f"{nombre} {var} {nombre_obj}: MAPE={mape_v:.2f}%")

predicciones.to_csv(RES / "predicciones_esquema_un_dia.csv")
pd.DataFrame(filas).to_csv(RES / "metricas_esquema_un_dia.csv", index=False)

### 10.2. Modelos ingenuos de referencia

Sin una referencia elemental, una cifra de error carece de significado. Se calculan dos modelos ingenuos, el que repite el valor del día anterior y el que repite el del mismo día de la semana anterior. Para el Objetivo 2 se añade una tercera referencia congelada, que repite la semana previa al episodio sin recibir ninguna observación posterior al 27 de octubre.

In [ ]:
serie = pd.read_csv(PROC / "serie_diaria_con_externas.csv",
                    parse_dates=["fecha"]).set_index("fecha").asfreq("D")


def mape(y_real, y_pred):
    mask = y_real.notna() & y_pred.notna()
    y, yhat = y_real[mask], y_pred[mask]
    return float(np.mean(np.abs((y - yhat) / y)) * 100)


for var in OBJETIVOS:
    lag1 = serie[var].shift(1)
    lag7 = serie[var].shift(7)
    
    print(var)
    print(f"  dia anterior       obj1: {mape(test1[var], lag1.loc[test1.index]):.2f}%"
          f"   obj2: {mape(test2[var], lag1.loc[test2.index]):.2f}%")
    print(f"  semana anterior    obj1: {mape(test1[var], lag7.loc[test1.index]):.2f}%"
          f"   obj2: {mape(test2[var], lag7.loc[test2.index]):.2f}%")

### 10.3. Fechas comunes a los cuatro modelos

La red LSTM no puede predecir aquellas fechas cuya ventana de contexto contiene algún día sin dato, de modo que evalúa un conjunto de fechas ligeramente menor que el resto de modelos. Para que la comparación sea homogénea, todas las métricas finales se calculan sobre las fechas comunes a los cuatro modelos.

In [ ]:
p_lstm = pd.read_csv(RES / "predicciones_lstm.csv",
                     parse_dates=["fecha"]).set_index("fecha")

faltan = test1.index.difference(p_lstm.index)
print("Dias evaluados por el LSTM en el objetivo 1:", int((p_lstm.index < INICIO_OBJ2).sum()))
print("Dias evaluados por el LSTM en el objetivo 2:", int((p_lstm.index >= INICIO_OBJ2).sum()))
print("Fechas del objetivo 1 ausentes en el LSTM:", len(faltan))
print(list(faltan.strftime("%Y-%m-%d")))

### 10.4. Efecto de restringir la evaluación a las fechas comunes



In [ ]:
p_sarimax = pd.read_csv(RES / "predicciones_sarimax.csv",
                        parse_dates=["fecha"]).set_index("fecha")
p_arima = pd.read_csv(RES / "predicciones_arima.csv",
                      parse_dates=["fecha"]).set_index("fecha")
comunes = test1.index.intersection(p_lstm.index)

for var in OBJETIVOS:
    sx = p_sarimax[f"pred_{var}_sin_conocimiento"]
    sa = p_arima[f"pred_{var}"]
    print(var)
    print(f"  SARIMAX  {len(test1)} fechas: {mape(test1[var], sx.reindex(test1.index)):.2f}%"
          f"   {len(comunes)} fechas: {mape(test1.loc[comunes, var], sx.reindex(comunes)):.2f}%")
    print(f"  SARIMA   {len(test1)} fechas: {mape(test1[var], sa.reindex(test1.index)):.2f}%"
          f"   {len(comunes)} fechas: {mape(test1.loc[comunes, var], sa.reindex(comunes)):.2f}%")

## 11. Evaluación comparativa

### 11.1. Tabla comparativa unificada



In [ ]:
p_prophet = pd.read_csv(RES / "predicciones_prophet.csv",
                        parse_dates=["fecha"]).set_index("fecha")
p_undia = pd.read_csv(RES / "predicciones_esquema_un_dia.csv",
                      parse_dates=["fecha"]).set_index("fecha")

comunes1 = test1.index.intersection(p_lstm.index)
comunes1 = comunes1[test1.loc[comunes1, OBJETIVOS].notna().all(axis=1)]
comunes2 = test2.index.intersection(p_lstm.index)
print(f"Fechas comunes objetivo 1: {len(comunes1)}   objetivo 2: {len(comunes2)}")

filas = []
for var in OBJETIVOS:
    fuentes = [
        ("SARIMA", "A", p_arima[f"pred_{var}"]),
        ("SARIMAX", "A", p_sarimax[f"pred_{var}_sin_conocimiento"]),
        ("Prophet", "A", p_prophet[f"pred_{var}_sin_conocimiento"]),
        ("SARIMA", "B", p_undia[f"pred_SARIMA_{var}"]),
        ("SARIMAX", "B", p_undia[f"pred_SARIMAX_{var}"]),
        ("Prophet", "B", p_prophet[f"pred_{var}_sin_conocimiento"]),
        ("LSTM", "B", p_lstm[f"pred_{var}"]),
        ("Ingenuo dia anterior", "B", serie[var].shift(1)),
        ("Ingenuo semana anterior", "B", serie[var].shift(7)),
    ]
    for nombre, esquema, pred in fuentes:
        for obj, idx, test_df in [("objetivo1", comunes1, test1),
                                  ("objetivo2", comunes2, test2)]:
            filas.append({"modelo": nombre, "esquema": esquema, "objetivo": obj,
                          "variable": var,
                          "MAPE": mape(test_df.loc[idx, var], pred.reindex(idx))})

tabla = pd.DataFrame(filas)
resumen = (tabla.groupby(["esquema", "modelo", "objetivo"])["MAPE"]
           .mean().unstack().round(2))
resumen["degradacion"] = (resumen["objetivo2"] / resumen["objetivo1"]).round(2)
print(resumen.to_string())

tabla.to_csv(RES / "tabla_comparativa_unificada.csv", index=False)
resumen.to_csv(RES / "resumen_comparativo.csv")

### 11.2. Desglose diario de residuos durante la ventana del episodio

El error porcentual con signo día a día permite distinguir entre modelos que sobrestiman la movilidad y modelos que la subestiman.

In [ ]:
for var in OBJETIVOS:
    real = test2[var]
    fuentes = {
        "SARIMA": p_undia[f"pred_SARIMA_{var}"],
        "SARIMAX": p_undia[f"pred_SARIMAX_{var}"],
        "Prophet": p_prophet[f"pred_{var}_sin_conocimiento"],
        "LSTM": p_lstm[f"pred_{var}"],
    }
    desglose = pd.DataFrame({"real": real.round(0)})
    for nombre, pred in fuentes.items():
        desglose[nombre] = ((real - pred.reindex(real.index)) / real * 100).round(1)
    print(f"Residuo porcentual con signo, {var}")
    print(desglose.to_string())
    print("Media por modelo:")
    print(desglose[list(fuentes)].mean().round(1).to_string())
    desglose.to_csv(RES / f"desglose_diario_{var}_esquema_b.csv")

### 11.3. Figura de residuos diarios

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for eje, var, titulo in zip(ejes, OBJETIVOS, ["Entradas", "Salidas"]):
    real = test2[var]
    fuentes = {
        "SARIMA": p_undia[f"pred_SARIMA_{var}"],
        "SARIMAX": p_undia[f"pred_SARIMAX_{var}"],
        "Prophet": p_prophet[f"pred_{var}_sin_conocimiento"],
        "LSTM": p_lstm[f"pred_{var}"],
    }
    for nombre, pred in fuentes.items():
        residuo = (real - pred.reindex(real.index)) / real * 100
        eje.plot(residuo.index, residuo.values, marker="o", label=nombre)
    eje.axhline(0, color="black", linewidth=0.8)
    eje.set_title(titulo)
    eje.set_ylabel("Residuo porcentual (%)")
    eje.grid(alpha=0.3)
    eje.legend(loc="lower left", ncol=4, fontsize=9)

ejes[1].set_xlabel("Fecha")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG / "residuos_diarios_esquema_b.png", dpi=300)
plt.show()

## 12. Variables de intensidad de precipitación

La variable indicador `es_dana` no tiene ningún precedente en el tramo de entrenamiento, por lo que resulta estructuralmente no identificable. Cabe preguntarse, no obstante, si una variable de intensidad de lluvia construida con precedentes en 2022 y 2023 habría permitido anticipar el episodio.

### 12.1. Construcción y evaluación

El umbral se fija con el percentil 95 de la precipitación del tramo de entrenamiento exclusivamente.

In [ ]:
serie = pd.read_csv(PROC / "serie_diaria_con_externas.csv",
                    parse_dates=["fecha"]).set_index("fecha").asfreq("D")

umbral = serie.loc[:FIN_TRAIN, "precipitacion"].quantile(0.95)
print(f"Umbral de lluvia intensa (percentil 95 del train): {umbral:.2f} mm")

serie["lluvia_intensa"] = (serie["precipitacion"] > umbral).astype(int)

print("Dias de lluvia intensa por anio:")
print(serie.groupby(serie.index.year)["lluvia_intensa"].sum())

serie.to_csv(PROC / "serie_diaria_con_intensidad.csv")

### 12.2. Aportación al SARIMAX



In [ ]:
CONJUNTOS = {
    "calendario": CALENDARIO,
    "calendario_intensidad": CALENDARIO + ["lluvia_intensa"],
}

train_i = serie.loc[:FIN_TRAIN]
test1_i = serie.loc[INICIO_OBJ1:FIN_OBJ1]
test2_i = serie.loc[INICIO_OBJ2:FIN_OBJ2]
test_total_i = pd.concat([test1_i, test2_i])

for nombre, cols in CONJUNTOS.items():
    print("Conjunto:", nombre)
    for var in OBJETIVOS:
        # Usamos tus mejores ordenes fijos
        p, d, q, P, D, Q = mejores_ordenes_sarimax[var]
        estado = SARIMAX(train_i[var], exog=train_i[cols],
                         order=(p, d, q), seasonal_order=(P, D, Q, ESTACIONALIDAD),
                         enforce_stationarity=False,
                         enforce_invertibility=False).fit(disp=False)
        if nombre == "calendario_intensidad":
            for c in ["lluvia_intensa"]:
                print(f"    coeficiente {c}: {estado.params.get(c, np.nan):.2f}"
                      f"   p-valor: {estado.pvalues.get(c, np.nan):.4f}")
        preds = []
        for fecha in test_total_i.index:
            exog_dia = test_total_i.loc[[fecha], cols]
            preds.append(float(estado.forecast(steps=1, exog=exog_dia).iloc[0]))
            estado = estado.append(test_total_i.loc[[fecha], var],
                                   exog=exog_dia, refit=False)
        pred = pd.Series(preds, index=test_total_i.index)
        print(f"  {var}  obj1: {mape(test1_i[var], pred.loc[test1_i.index]):.2f}%"
              f"   obj2: {mape(test2_i[var], pred.loc[test2_i.index]):.2f}%")

### 12.3. Comportamiento del indicador durante la ventana



In [ ]:
ventana = serie.loc[INICIO_OBJ2:FIN_OBJ2,
                    ["precipitacion", "precip_acum_3d", "lluvia_intensa",
                     "es_festivo", "entradas_viajes"]]
print(ventana.to_string())
print("Dias con lluvia intensa en la ventana:", int(ventana["lluvia_intensa"].sum()))

### 12.4. Variante retardada



In [ ]:
serie_i = pd.read_csv(PROC / "serie_diaria_con_intensidad.csv",
                      parse_dates=["fecha"]).set_index("fecha").asfreq("D")
serie_i["lluvia_intensa_lag1"] = serie_i["lluvia_intensa"].shift(1).fillna(0)

train_i = serie_i.loc[:FIN_TRAIN]
test1_i = serie_i.loc[INICIO_OBJ1:FIN_OBJ1]
test2_i = serie_i.loc[INICIO_OBJ2:FIN_OBJ2]
test_total_i = pd.concat([test1_i, test2_i])

CONJUNTOS_LAG = {
    "calendario": CALENDARIO,
    "intensa": CALENDARIO + ["lluvia_intensa"],
    "intensa_retardada": CALENDARIO + ["lluvia_intensa_lag1"],
    "intensa_ambas": CALENDARIO + ["lluvia_intensa", "lluvia_intensa_lag1"],
}

VAR = "entradas_viajes"
p, d, q, P, D, Q = ordenes_sarimax[VAR]

errores = pd.DataFrame(index=test2_i.index)
errores["real"] = test2_i[VAR].round(0)

for nombre, cols in CONJUNTOS_LAG.items():
    estado = SARIMAX(train_i[VAR], exog=train_i[cols],
                     order=(p, d, q), seasonal_order=(P, D, Q, ESTACIONALIDAD),
                     enforce_stationarity=False,
                     enforce_invertibility=False).fit(disp=False)
    for c in cols:
        if c.startswith("lluvia"):
            print(f"{nombre:20s} coeficiente {c:22s}"
                  f"{estado.params.get(c, np.nan):>12.0f}  "
                  f"p={estado.pvalues.get(c, np.nan):.4f}")
    preds = []
    for fecha in test_total_i.index:
        exog_dia = test_total_i.loc[[fecha], cols]
        preds.append(float(estado.forecast(steps=1, exog=exog_dia).iloc[0]))
        estado = estado.append(test_total_i.loc[[fecha], VAR],
                               exog=exog_dia, refit=False)
    pred = pd.Series(preds, index=test_total_i.index)
    errores[nombre] = ((pred.loc[test2_i.index] - test2_i[VAR])
                       / test2_i[VAR] * 100).round(1)
    print(f"{nombre:20s} obj1: {mape(test1_i[VAR], pred.loc[test1_i.index]):.2f}%"
          f"   obj2: {mape(test2_i[VAR], pred.loc[test2_i.index]):.2f}%")

print("Error porcentual con signo por dia en la ventana:")
print(errores.to_string())

## 13. Caracterización del episodio como evento anómalo

### 13.1. Desviación frente a la línea base estacional

La línea base se construye con el mismo día de la semana en los meses de octubre y noviembre del tramo de entrenamiento, excluyendo días faltantes y festivos

In [ ]:
df = pd.read_csv(PROC / "serie_diaria_con_externas.csv",
                 parse_dates=["fecha"]).set_index("fecha")

base = df.loc[FECHA_INICIO:FIN_TRAIN]
base = base[base.index.month.isin([10, 11])
            & (base["es_faltante"] == 0)
            & (base["es_festivo"] == 0)]
medias_dow = base.groupby(base.index.dayofweek)[OBJETIVOS].mean()

dana = df.loc[INICIO_OBJ2:FIN_OBJ2, OBJETIVOS].copy()
for var in OBJETIVOS:
    dana[f"base_{var}"] = [medias_dow.loc[d, var] for d in dana.index.dayofweek]
    dana[f"desv_{var}_pct"] = ((dana[var] - dana[f"base_{var}"])
                               / dana[f"base_{var}"] * 100)

tabla_desv = dana[[f"desv_{v}_pct" for v in OBJETIVOS]].round(1)
tabla_desv.columns = ["Desviacion entradas (%)", "Desviacion salidas (%)"]
print(tabla_desv.to_string())

print("Dias de referencia por dia de la semana:")
print(base.groupby(base.index.dayofweek).size().to_string())

### 13.2. Desviación por componente de flujo



In [ ]:
externa = pd.read_csv(PROC / "serie_diaria_externa.csv",
                      parse_dates=["fecha"]).set_index("fecha").asfreq("D")
agregados = pd.read_csv(PROC / "agregados_od_separados.csv",
                        parse_dates=["fecha"]).set_index("fecha")
COMPONENTES = ["internos", "entradas_externas", "salidas_externas"]

comp = agregados.join(externa[["es_festivo", "es_faltante", "es_dana"]])
base = comp.loc[FECHA_INICIO:FIN_TRAIN]
base = base[base.index.month.isin([10, 11])
            & (base["es_faltante"] == 0)
            & (base["es_festivo"] == 0)]
medias = base.groupby(base.index.dayofweek)[COMPONENTES].mean()

dana_comp = comp.loc[INICIO_OBJ2:FIN_OBJ2].copy()
for c in COMPONENTES:
    dana_comp[f"desv_{c}"] = [
        (dana_comp[c].iloc[i] - medias.loc[d, c]) / medias.loc[d, c] * 100
        for i, d in enumerate(dana_comp.index.dayofweek)]

print("Desviacion por componente frente a la linea base estacional (%):")
print(dana_comp[[f"desv_{c}" for c in COMPONENTES]].round(1).to_string())
print("Festivos dentro de la ventana:", dana_comp[dana_comp["es_festivo"] == 1].index.date)

dana_comp.to_csv(RES_B2 / "desviacion_por_componente.csv")

### 13.3. Detección de anomalías mediante residuos estandarizados



In [ ]:
UMBRAL_Z = 3.0

df_anom = pd.read_csv(PROC / "serie_diaria_con_externas.csv",
                      parse_dates=["fecha"]).set_index("fecha").asfreq("D")
exog = df_anom[EXTERNAS]

anom = pd.DataFrame(index=df_anom.index)
for var in OBJETIVOS:
    res = SARIMAX(df_anom[var], exog=exog, order=(1, 0, 2),
                  seasonal_order=(0, 1, 1, ESTACIONALIDAD),
                  enforce_stationarity=False,
                  enforce_invertibility=False).fit(disp=False)
    r = res.resid.iloc[8:]
    anom[f"z_{var}"] = (r - r.mean()) / r.std()
    anom[f"anom_{var}"] = (anom[f"z_{var}"].abs() > UMBRAL_Z).fillna(False)

anom["es_anomalo"] = (anom["anom_entradas_viajes"]
                      | anom["anom_salidas_viajes"]).astype(int)
anom["es_dana"] = df_anom["es_dana"]

print("Umbral z:", UMBRAL_Z)
print("Total de dias anomalos 2022-2024:", int(anom["es_anomalo"].sum()))
print("Dias de la ventana marcados como anomalos:",
      int(anom.loc[anom["es_dana"] == 1, "es_anomalo"].sum()), "de 8")
print(anom.loc[anom["es_anomalo"] == 1,
               ["z_entradas_viajes", "z_salidas_viajes", "es_dana"]].round(2).to_string())

anom.to_csv(RES / "anomalias_zscore.csv")